In [523]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures

In [524]:
seed = 42

# Data

In [525]:
def clean_df(df: pd.DataFrame):
    if df["BestSquatKg"].dtype == 'object':
        df["BestSquatKg"] = df['BestSquatKg'].str.replace("..", ".")
        df["BestSquatKg"] = df["BestSquatKg"].astype(np.float32)

    df["BodyweightKg"] = np.abs(df["BodyweightKg"])
    df["BestSquatKg"] = np.abs(df["BestSquatKg"])
    df["BestDeadliftKg"] = np.abs(df["BestDeadliftKg"])

    if "BestBenchKg" in df:
        df["BestBenchKg"] = np.abs(df["BestBenchKg"])

    return df

def prep_df(df: pd.DataFrame):
    df = df.drop(["playerId", "BestBenchKg", "Name"], axis=1, errors='ignore')

    # fix missing
    df['Age'] = df['Age'].fillna(df['Age'].mean())

    # dummy encoding
    dummy_cols = df.select_dtypes(include=['object'])
    for col in dummy_cols:
        dummies = pd.get_dummies(df[col], col, drop_first=True)
        df = pd.concat([df, dummies], axis=1)
    df = df.select_dtypes(exclude='object')

    return df

In [526]:
df = pd.read_csv("train_data.csv")
df = clean_df(df)
train_df = prep_df(df)

In [527]:
train_df.head()

,Age,BodyweightKg,BestSquatKg,BestDeadliftKg,Sex_M,Equipment_Raw,Equipment_Single-ply,Equipment_Wraps
0,23.0,87.30,205.0,235.0,True,True,False,False
1,23.0,73.48,220.0,260.0,True,False,False,True
2,26.0,112.40,142.5,220.0,True,True,False,False
3,35.0,59.42,95.0,102.5,False,True,False,False
4,26.5,61.40,105.0,127.5,False,True,False,False


# EDA

In [528]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18900 entries, 0 to 18899
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   playerId        18900 non-null  float64
 1   Name            18900 non-null  object 
 2   Sex             18900 non-null  object 
 3   Equipment       18900 non-null  object 
 4   Age             18725 non-null  float64
 5   BodyweightKg    18900 non-null  float64
 6   BestSquatKg     18900 non-null  float32
 7   BestDeadliftKg  18900 non-null  float64
 8   BestBenchKg     18900 non-null  float64
dtypes: float32(1), float64(5), object(3)
memory usage: 1.2+ MB


In [529]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18900 entries, 0 to 18899
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Age                   18900 non-null  float64
 1   BodyweightKg          18900 non-null  float64
 2   BestSquatKg           18900 non-null  float32
 3   BestDeadliftKg        18900 non-null  float64
 4   Sex_M                 18900 non-null  bool   
 5   Equipment_Raw         18900 non-null  bool   
 6   Equipment_Single-ply  18900 non-null  bool   
 7   Equipment_Wraps       18900 non-null  bool   
dtypes: bool(4), float32(1), float64(3)
memory usage: 590.8 KB


# Model selection

In [530]:
X_train, X_test, y_train, y_test = train_test_split(train_df, df["BestBenchKg"], test_size=0.33, random_state=seed)

In [531]:
def evaluate(clf):
    scores = cross_val_score(clf, X_train, y_train, scoring='r2', cv=3, n_jobs=-1)

    clf.fit(X_train, y_train)
    score = r2_score(y_test, clf.predict(X_test))
    
    return (np.mean(scores) - np.std(scores)).item(), score

In [532]:
lrr = Ridge()

evaluate(lrr)

(0.8576373468299984, 0.8618571907170671)

In [533]:
lr = Pipeline([
    ('scale', PolynomialFeatures()),
    ('clf',  Ridge())
])

evaluate(lr)

(0.8732182556199319, 0.8718097442448975)

In [534]:
clf = lr

clf.fit(X_train, y_train)

Pipeline(steps=[('scale', PolynomialFeatures()), ('clf', Ridge())])

# Submission

In [535]:
test_df_ = pd.read_csv("test_data.csv")
test_df_ = clean_df(test_df_)
test_df = prep_df(test_df_)

In [536]:
# subtask 1
total = df["BestSquatKg"] + df["BestDeadliftKg"] + df["BestBenchKg"]
subtask1 = ((total / df["BodyweightKg"]) > 5).sum().astype(np.int32)

# subtask 2
subtask2 = clf.predict(test_df)

In [537]:
def build_subtask(sid, answers):
    return pd.DataFrame({
        "subtaskID": sid,
        "datapointID": test_df_["playerId"] if sid == 2 else [1],
        "answer": answers
    })

subtasks = [
    (1, subtask1),
    (2, subtask2)
]

submission = pd.concat([build_subtask(sid, ans) for (sid, ans) in subtasks])
submission["datapointID"] = submission["datapointID"].astype(np.int32)
submission["answer"] = submission["answer"].astype(np.int32)

In [538]:
submission.head()

,subtaskID,datapointID,answer
0,1,1,13224
0,2,2308,124
1,2,22404,95
2,2,23397,191
3,2,25058,127


In [539]:
submission.to_csv("submission.csv", index=False)